In [ ]:
import pandas as pd
import platform
import psutil
from datetime import datetime
import subprocess
import GPUtil
import re
import os
from openpyxl import load_workbook
import math
import pyautogui
import time

def appendDataPointToDatabase(dataList, excelFilePath, sheetName):
    try:
        # Load the existing workbook
        book = load_workbook(excelFilePath)
        # Select the specified sheet
        sheet = book[sheetName]
        # Append the new data as a row
        sheet.append(dataList)
        # Save the file
        book.save(excelFilePath)
        print("Data successfully appended to Excel file.")
    except Exception as e:
        print(f"An error occurred: {e}")

def extractPerformanceMetrics(output):
    # Split the output into lines
    lines = output.splitlines()

    # Initialize variables to store extracted values
    setupTime, algorithmTime, optimTime, ramUsage, threadCount = None, None, None, None, None

    # Use regex to find and extract the values
    for line in lines:
        if "Setup Time" in line:
            setupTime = float(re.search(r"[-+]?\d*\.\d+|\d+", line).group())  # Extract the float number
        elif "Algorithm Computation Time" in line:
            algorithmTime = float(re.search(r"[-+]?\d*\.\d+|\d+", line).group())  # Extract the float number
        elif "Optimization Time" in line:
            optimTime = float(re.search(r"[-+]?\d*\.\d+|\d+", line).group())  
        elif "RAM Usage" in line:
            ramUsage = float(re.search(r"[-+]?\d*\.\d+|\d+", line).group())  # Extract the float number
        elif "Threads Used" in line:
            threadCount = float(re.search(r"[-+]?\d*\.\d+|\d+", line).group())

    return setupTime, algorithmTime, optimTime, ramUsage, threadCount

def generateUniqueId(excelFilePath, sheetName): 
    # Check if the Excel file exists
    if os.path.exists(excelFilePath):
        # Load the existing Excel file and sheet
        excelData = pd.read_excel(excelFilePath, sheet_name=sheetName)

        # Check if the sheet has any data
        if len(excelData) > 0:
            # Get the last UniqueID
            lastUid = excelData['Unique ID'].iloc[-1]
            # Increment the numeric part of the ID by 1
            lastIdNum = int(lastUid[2:])  # Extract numeric part (remove "JL")
            newIdNum = lastIdNum + 1  # Increment by 1
            newUid = f"JL{newIdNum:06d}"  # Zero-pad the new ID to maintain 6 digits
        else:
            # If the sheet exists but has no data
            newUid = "JL000000"
    else:
        # If the file does not exist
        newUid = "JL000000"

    return newUid

def getLastBatchId(excelFilePath, sheetName):
    # Load the Excel file
    book = load_workbook(excelFilePath)
    sheet = book[sheetName]

    # Get all BatchIDs from the file
    batchIds = []
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row, min_col=2, max_col=2):
        if row and row[0].value:  # Ensure there is at least one element in row and it's not None
            batchIds.append(row[0].value)

    # Check if there are no batch IDs found
    if not batchIds:
        return "Batch000"

    # Extract the numerical part and return the highest BatchID
    lastBatchNumber = int(batchIds[-1].replace("Batch", ""))
    return f"Batch{str(lastBatchNumber).zfill(3)}"

def prepareDataPoint(juliaResultString, excelFilePath, sheetName, batchId, runId):
    uid = generateUniqueId(excelFilePath, sheetName)  # Generate or retrieve UniqueID

    # Automatically detect system information
    cpu = platform.processor()  # Get the CPU information
    ram = math.ceil(psutil.virtual_memory().total / (1024 ** 3))  # Get total RAM in GB
    date = datetime.now().strftime("%m/%d/%Y")  # Get the current date

    # Detect GPU information using GPUtil
    gpus = GPUtil.getGPUs()  # Get list of all GPUs
    if gpus:
        gpu = ', '.join([f"{gpu.name}" for gpu in gpus])
    else:
        gpu = "No compatible GPU detected"

    # Get Julia version
    juliaVersion = subprocess.run(['julia', '--version'], capture_output=True, text=True, encoding='utf-8', shell=True)
    if juliaVersion.returncode == 0:
        langVer = juliaVersion.stdout.strip().split()[-1]  # Extract version number from output
    else:
        langVer = "Unable to detect Julia Mode."

    # Manually set other details
    computeMedia = "CPU"
    algorithmType = "Gradient Descent"

    # Process and extract performance metrics
    preprocessTime, algorithmTime, optimTime, ramUsage, coresUsed = extractPerformanceMetrics(juliaResultString)
    totalRunTime = float(preprocessTime) + float(algorithmTime) + float(optimTime)

    # Determine concurrency mode
    concurrencyMode = "Single-Core" if int(coresUsed) == 1 else "Multi-Core"

    # Create the data row
    dataRow = [
        uid, batchId, runId, date, langVer, cpu, ram, gpu, computeMedia,
        concurrencyMode, coresUsed, algorithmType, preprocessTime,
        algorithmTime, optimTime, totalRunTime, ramUsage
    ]

    return dataRow

def callStartupJuliaFile(runIndex):
    # Use pyautogui to call Julia script in PowerShell
    pyautogui.typewrite(f'include("S:\\\\Research Material\\\\CSE\\\\MU\\\\Project\\\\1Code\\\\AlgorithmAnalysis.jl\\\\1CPU\\\\src\\\\startupCPU.jl")')
    pyautogui.press('enter')
    
    # Adjust sleep time based on the run index
    if runIndex == 0:
        time.sleep(17)  # Longer sleep time for the first run to account for JIT compilation
    else:
        time.sleep(3)

    # Check if the output file exists, wait until it's created or timeout
    outputFilePath = "S:\\Research Material\\CSE\\MU\\Project\\5Testing\\output.txt"
    timeout = 30  # Set a maximum timeout duration (in seconds)
    startTime = time.time()
    
    # Loop to wait for the file or timeout
    while not os.path.exists(outputFilePath):
        # Check if timeout has been reached
        if time.time() - startTime > timeout:
            print("Timeout reached while waiting for Julia to create the output file.")
            return None
        
        print("Waiting for Julia to create the output file...")
        time.sleep(1)

    # Read output from the file
    with open(outputFilePath, 'r') as file:
        output = file.read()
    
    # Delete the file after reading
    os.remove(outputFilePath)

    return output
    
def runAlgorithmInJuliaNTimes(numberOfBatches, runsPerBatch, excelFilePath, sheetName):
    # Determine the starting BatchID once at the beginning
    lastBatchId = getLastBatchId(excelFilePath, sheetName)
    batchCounter = int(lastBatchId.replace("Batch", ""))

    for batchIndex in range(numberOfBatches):
        # Start a new PowerShell window and Julia REPL for the current batch
        print(f"Starting new batch {batchIndex + 1}...")
        process = subprocess.Popen('powershell.exe', creationflags=subprocess.CREATE_NEW_CONSOLE)
        time.sleep(4)  # Wait for the PowerShell window to open
        pyautogui.typewrite('julia')
        pyautogui.press('enter')
        time.sleep(2)  # Give time for Julia to start

        # Increment batch number for the current batch
        batchId = f"Batch{str(batchCounter + 1).zfill(3)}"
        
        for runIndex in range(runsPerBatch):
            if runIndex == 0:
                # First run in a new batch (includes JIT compilation overhead)
                print(f"Running batch {batchId} run {runIndex + 1} with JIT compilation overhead...")
            else:
                # Subsequent runs in the same batch
                print(f"Running batch {batchId} run {runIndex + 1} without JIT compilation overhead...")

            # Call the Julia script and get the result
            juliaResultString = callStartupJuliaFile(runIndex)
            if juliaResultString:
                # Generate Run ID for the current run
                runId = runIndex
                # Prepare data point for this runexit()
                
                newDataPoint = prepareDataPoint(juliaResultString, excelFilePath, sheetName, batchId, runId)
                
                # Append the data point to the Excel file
                appendDataPointToDatabase(newDataPoint, excelFilePath, sheetName)
            else:
                print(f"Failed to execute Julia script for batch {batchId} run {runIndex + 1}.")

        # Increment batch counter after completing the current batch
        batchCounter += 1

        # Close the Julia REPL by sending the exit command
        pyautogui.typewrite('exit()')
        pyautogui.press('enter')
        time.sleep(3)  # Allow time for Julia to close

        # Close the PowerShell or Command Prompt window
        pyautogui.hotkey('alt', 'f4')
        time.sleep(3)  # Wait a bit to ensure the terminal is closed properly

        print(f"Batch {batchIndex + 1} completed and Julia session closed.")

def main():
    excelFilePath = "S:\\Research Material\\CSE\\MU\\Project\\2Database\\Database.xlsx"
    sheetName     = "Julia"

    # Ask user for the number of batches and runs per batch
    numberOfBatches = 3
    runsPerBatch    = 3

    # Run the algorithm in Julia as specified
    runAlgorithmInJuliaNTimes(numberOfBatches, runsPerBatch, excelFilePath, sheetName)

if __name__ == "__main__":
    main()